In [27]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer

In [26]:
from tensorflow.keras.preprocessing.text import Tokenizer
import pandas as pd

tokenizer = Tokenizer(num_words=10000)

# First pass to fit tokenizer
chunksize = 1000
for chunk in pd.read_csv('shortjokes.csv', chunksize=chunksize):
    texts = chunk['Joke'].astype(str).tolist()
    tokenizer.fit_on_texts(texts)  # Learn vocab

In [28]:
from tensorflow.keras.preprocessing.sequence import pad_sequences



for chunk in pd.read_csv('shortjokes.csv', chunksize=chunksize):
    texts = chunk['Joke'].astype(str).tolist()
    sequences = tokenizer.texts_to_sequences(texts)
    padded_input_sequences = pad_sequences(sequences, maxlen=100, padding='pre')  # Adjust maxlen


In [29]:
from itertools import islice
dict(islice(tokenizer.word_index.items(),5) ) # Dict: word → index


{'a': 1, 'the': 2, 'i': 3, 'to': 4, 'you': 5}

In [30]:
# tokenizer.texts_to_sequences(texts)  # List of lists of token indices

In [31]:
len(tokenizer.word_index)

70648

In [32]:
input_sequences = []
for sentence in chunk['Joke']:
  tokenized_sentence = tokenizer.texts_to_sequences([sentence])[0]

  for i in range(1,len(tokenized_sentence)):
    input_sequences.append(tokenized_sentence[:i+1])

In [33]:
input_sequences[:5]

[[3, 20],
 [3, 20, 2801],
 [3, 20, 2801, 18],
 [3, 20, 2801, 18, 92],
 [3, 20, 2801, 18, 92, 1731]]

In [34]:
max_len=max([len(x) for x in input_sequences])

In [35]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
padded_input_sequences = pad_sequences(input_sequences, maxlen = max_len, padding='pre')

In [36]:
padded_input_sequences

array([[   0,    0,    0, ...,    0,    3,   20],
       [   0,    0,    0, ...,    3,   20, 2801],
       [   0,    0,    0, ...,   20, 2801,   18],
       ...,
       [   0,    0,    0, ...,   35, 9892,  925],
       [   0,    0,    0, ..., 9892,  925,    8],
       [   0,    0,    0, ...,  925,    8,  637]], dtype=int32)

In [37]:
X = padded_input_sequences[:,:-1]

In [38]:
y = padded_input_sequences[:,-1]

In [39]:
X.shape

(10109, 37)

In [40]:
y.shape

(10109,)

In [41]:
len(tokenizer.word_index)

70648

In [42]:
from tensorflow.keras.utils import to_categorical
y = to_categorical(y,num_classes=70649) #one hot encoding

In [43]:
y.shape

(10109, 70649)

In [44]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, GRU, Dense, Dropout

# Define model
model = Sequential()
model.add(Embedding(input_dim=70649, output_dim=100, input_length=max_len))
model.add(Bidirectional(GRU(128, return_sequences=True)))
model.add(Dropout(0.3))
model.add(Bidirectional(GRU(128)))
model.add(Dropout(0.3))
model.add(Dense(70649, activation='softmax'))

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [45]:
model.compile(loss='categorical_crossentropy', optimizer='adam',metrics=['accuracy'])

In [46]:
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.models import load_model
import pickle

# Save the best model (lowest val_loss)
checkpoint_cb = ModelCheckpoint(
    'model_checkpoint.h5',
    save_best_only=True,
    monitor='val_loss',
    mode='min',
    verbose=1
)

# Train the model
model.fit(
    X, y,
    validation_split=0.2,
    epochs=10,
    callbacks=[checkpoint_cb]
)

# Load the best model and save it as a .pkl file
best_model = load_model('model_checkpoint.h5')
with open('best_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)


Epoch 1/10
252/253 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.0359 - loss: 8.9475
Epoch 1: val_loss improved from inf to 7.61255, saving model to model_checkpoint.h5


253/253 ━━━━━━━━━━━━━━━━━━━━ 30s 98ms/step - accuracy: 0.0359 - loss: 8.9411 - val_accuracy: 0.0425 - val_loss: 7.6125
Epoch 2/10
252/253 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.0429 - loss: 6.9727
Epoch 2: val_loss improved from 7.61255 to 7.60429, saving model to model_checkpoint.h5


253/253 ━━━━━━━━━━━━━━━━━━━━ 35s 75ms/step - accuracy: 0.0429 - loss: 6.9730 - val_accuracy: 0.0539 - val_loss: 7.6043
Epoch 3/10
253/253 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.0488 - loss: 6.5927
Epoch 3: val_loss did not improve from 7.60429
253/253 ━━━━━━━━━━━━━━━━━━━━ 16s 57ms/step - accuracy: 0.0488 - loss: 6.5929 - val_accuracy: 0.0549 - val_loss: 7.6593
Epoch 4/10
253/253 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.0596 - loss: 6.2181
Epoch 4: val_loss did not improve from 7.60429
253/253 ━━━━━━━━━━━━━━━━━━━━ 19s 52ms/step - accuracy: 0.0596 - loss: 6.2185 - val_accuracy: 0.0603 - val_loss: 7.6111
Epoch 5/10
252/253 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.0667 - loss: 5.9659
Epoch 5: val_loss improved from 7.60429 to 7.53121, saving model to model_checkpoint.h5


253/253 ━━━━━━━━━━━━━━━━━━━━ 26s 73ms/step - accuracy: 0.0667 - loss: 5.9664 - val_accuracy: 0.0663 - val_loss: 7.5312
Epoch 6/10
252/253 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.0750 - loss: 5.7299
Epoch 6: val_loss did not improve from 7.53121
253/253 ━━━━━━━━━━━━━━━━━━━━ 13s 53ms/step - accuracy: 0.0750 - loss: 5.7302 - val_accuracy: 0.0653 - val_loss: 7.7203
Epoch 7/10
253/253 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.0855 - loss: 5.4825
Epoch 7: val_loss did not improve from 7.53121
253/253 ━━━━━━━━━━━━━━━━━━━━ 20s 53ms/step - accuracy: 0.0855 - loss: 5.4827 - val_accuracy: 0.0732 - val_loss: 7.8002
Epoch 8/10
253/253 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.0999 - loss: 5.2295
Epoch 8: val_loss did not improve from 7.53121
253/253 ━━━━━━━━━━━━━━━━━━━━ 21s 53ms/step - accuracy: 0.0999 - loss: 5.2297 - val_accuracy: 0.0767 - val_loss: 7.8024
Epoch 9/10
252/253 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.1174 - loss: 4.9478
Epoch 9: val_loss did not improve f

In [47]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 37, 100)        │     7,064,900 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 37, 256)        │       176,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 37, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ (None, 256)            │       296,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 70649)          │    18,156,793 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 77,084,345 (294.05 MB)

 Trainable params: 25,694,781 (98.02 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 51,389,564 (196.04 MB)

In [55]:
text = "small "
import numpy as np

for i in range(10):
  # tokenize
  token_text = tokenizer.texts_to_sequences([text])[0]
  # padding
  padded_token_text = pad_sequences([token_text], maxlen=38, padding='pre')
  # predict
  def sample_with_temperature(preds, temperature=1.0):
    preds = np.asarray(preds).astype("float64")
    preds = np.log(preds + 1e-10) / temperature
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)
    probas = np.random.multinomial(1, preds, 1)
    return np.argmax(probas)

  pred = model.predict(padded_token_text, verbose=0)[0]
  pos = sample_with_temperature(pred, temperature=0.8)

  for word,index in tokenizer.word_index.items():
    if index == pos:
      text = text + " " + word
      print(text)

small  more
small  more walking
small  more walking out
small  more walking out me
small  more walking out me with
small  more walking out me with honey
small  more walking out me with honey 3
small  more walking out me with honey 3 have
small  more walking out me with honey 3 have the
small  more walking out me with honey 3 have the talking


In [49]:
with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

In [50]:
'''from tensorflow.keras.models import load_model

# Load the best saved model
model = load_model('model_checkpoint.h5')

# Resume training for more epochs
model.fit(
    X, y,
    validation_split=0.2,
    epochs=5,  # train for 5 more epochs
    callbacks=[checkpoint_cb]
)'''

"from tensorflow.keras.models import load_model\n\n# Load the best saved model\nmodel = load_model('model_checkpoint.h5')\n\n# Resume training for more epochs\nmodel.fit(\n    X, y,\n    validation_split=0.2,\n    epochs=5,  # train for 5 more epochs\n    callbacks=[checkpoint_cb]\n)"